In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATA_PATH = r"C:\Users\MEGHANA\OneDrive\Desktop\SmartDine\data\raw\zomato.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (51717, 17)


In [3]:
print("Columns in the dataset:")
print()

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Columns in the dataset:

1. url
2. address
3. name
4. online_order
5. book_table
6. rate
7. votes
8. phone
9. location
10. rest_type
11. dish_liked
12. cuisines
13. approx_cost(for two people)
14. reviews_list
15. menu_item
16. listed_in(type)
17. listed_in(city)


In [4]:
duplicate_counts = (
    df.groupby(["name", "address"])
      .size()
      .sort_values(ascending=False)
)

print("Top 10 most repeated name + address combinations:")
print(duplicate_counts.head(10))

Top 10 most repeated name + address combinations:
name                               address                                                                          
Hammered                           18, Ali Askar Road, Cunningham Road, Bangalore                                       34
Dolci Desserts                     18/3, Ali Askar Road, Near Kingfisher Airlines Office, Cunningham Road, Bangalore    25
The Chocolate Heaven               G8/9/10, Ashoka Plaza, Kamaraj Road, Commercial Street, Bangalore                    24
Tiger Trail - Ramada Hotel         Ramada Hotel, 11, Park Road, Shivajinagar, Bangalore                                 23
ThrowBack - Pub & Dine             5th Floor, Confident Hotel, Hosur Main Road, Langford Town, Bangalore                22
Berrylicious Cafe                  4/1, Brunton Road, MG Road, Bangalore                                                22
Chillout                           8, G Cross, 20th Main Road, BTM 1st stage, BTM, Bangalore   

In [5]:
cols = [
    "name",
    "address",
    "location",
    "listed_in(city)",
    "listed_in(type)",
    "rate",
    "votes",
    "cuisines",
    "approx_cost(for two people)",
    "online_order",
    "book_table"
]

top_10_duplicates = duplicate_counts.head(10)

for (name, address), count in top_10_duplicates.items():

    print("\n" + "=" * 100)
    print(f"RESTAURANT: {name}")
    print(f"ADDRESS: {address}")
    print(f"NUMBER OF RECORDS: {count}")
    print("=" * 100)

    repeated_rows = df[
        (df["name"] == name) &
        (df["address"] == address)
    ][cols]

    print(repeated_rows.to_string(index=False))


RESTAURANT: Hammered
ADDRESS: 18, Ali Askar Road, Cunningham Road, Bangalore
NUMBER OF RECORDS: 34
    name                                        address        location listed_in(city)    listed_in(type)   rate  votes                                        cuisines approx_cost(for two people) online_order book_table
Hammered 18, Ali Askar Road, Cunningham Road, Bangalore Cunningham Road    Brigade Road              Cafes  4.7/5   4639 North Indian, Thai, Japanese, Continental, Cafe                       1,300          Yes        Yes
Hammered 18, Ali Askar Road, Cunningham Road, Bangalore Cunningham Road    Brigade Road           Delivery  4.7/5   4657 North Indian, Thai, Japanese, Continental, Cafe                       1,300          Yes        Yes
Hammered 18, Ali Askar Road, Cunningham Road, Bangalore Cunningham Road    Brigade Road           Desserts  4.7/5   4657 North Indian, Thai, Japanese, Continental, Cafe                       1,300          Yes        Yes
Hammered 18, Ali

In [6]:
# Columns we want to investigate for consistency
check_columns = [
    "location",
    "listed_in(city)",
    "listed_in(type)",
    "rate",
    "votes",
    "cuisines",
    "approx_cost(for two people)",
    "online_order",
    "book_table"
]

# Calculate number of unique values for each column
# within each name + address group
variation_summary = (
    df.groupby(["name", "address"])[check_columns]
      .nunique(dropna=False)
)

print("Variation summary:")
print(variation_summary.head(20))

Variation summary:
                                                                                                                                        location  \
name                                 address                                                                                                       
#FeelTheROLL                         Opposite Mantri Commercio, Outer Ring Road, Devarabisanahalli, Near Sakra World Hospital                  1   
#L-81 Cafe                           Sector 6, HSR Layout, HSR                                                                                 1   
#Vibes Restro                        Marasur Gate, Chandapura - Anekal Road, Near Aliance Collage, Electronic City, Bangalore                  1   
#refuel                              7, Ground Floor, RR Commercial Complex, Akshay Nagar, Bannerghatta Road, Bangalore                        1   
'Brahmins' Thatte Idli               19, 1st main, 2nd cross, 3rd stage, 3rd block, Basaveshw

In [7]:
# Count how many restaurant entities have variation
# in each column

variation_counts = (variation_summary > 1).sum()

variation_percentage = (
    variation_counts / len(variation_summary) * 100
).sort_values(ascending=False)

variation_report = pd.DataFrame({
    "entities_with_variation": variation_counts,
    "percentage_of_entities": variation_percentage.round(2)
})

print("Variation across name + address entities:")
print(variation_report)

Variation across name + address entities:
                             entities_with_variation  percentage_of_entities
approx_cost(for two people)                      409                    3.27
book_table                                        26                    0.21
cuisines                                         541                    4.33
listed_in(city)                                 8563                   68.51
listed_in(type)                                 7093                   56.75
location                                          30                    0.24
online_order                                     394                    3.15
rate                                            5547                   44.38
votes                                           3936                   31.49


In [8]:
important_columns = [
    "location",
    "rate",
    "cuisines",
    "approx_cost(for two people)",
    "online_order",
    "book_table"
]

consistency_report = []

for column in important_columns:

    # Number of unique values per restaurant entity
    unique_counts = (
        df.groupby(["name", "address"])[column]
          .nunique(dropna=False)
    )

    consistent = (unique_counts == 1).sum()
    inconsistent = (unique_counts > 1).sum()

    consistency_report.append({
        "column": column,
        "consistent_entities": consistent,
        "inconsistent_entities": inconsistent,
        "total_entities": len(unique_counts),
        "consistency_percentage": round(
            consistent / len(unique_counts) * 100, 2
        )
    })

consistency_df = pd.DataFrame(consistency_report)

print(consistency_df.to_string(index=False))

                     column  consistent_entities  inconsistent_entities  total_entities  consistency_percentage
                   location                12469                     30           12499                   99.76
                       rate                 6952                   5547           12499                   55.62
                   cuisines                11958                    541           12499                   95.67
approx_cost(for two people)                12090                    409           12499                   96.73
               online_order                12105                    394           12499                   96.85
                 book_table                12473                     26           12499                   99.79


In [9]:
# Find restaurant entities where rating has more than one value
rate_variation = (
    df.groupby(["name", "address"])["rate"]
      .nunique(dropna=False)
)

rate_variable_entities = rate_variation[rate_variation > 1]

print("Number of entities with varying rate values:",
      len(rate_variable_entities))

print("\nPercentage of entities with varying rate values:",
      round(len(rate_variable_entities) / len(rate_variation) * 100, 2))

Number of entities with varying rate values: 5547

Percentage of entities with varying rate values: 44.38


In [10]:
# Inspect the first 10 entities with varying ratings

for (name, address), _ in rate_variable_entities.head(10).items():

    print("\n" + "=" * 100)
    print("RESTAURANT:", name)
    print("ADDRESS:", address)
    print("=" * 100)

    temp = df[
        (df["name"] == name) &
        (df["address"] == address)
    ][["rate", "votes", "listed_in(city)", "listed_in(type)"]]

    print(temp.to_string(index=False))


RESTAURANT: #FeelTheROLL
ADDRESS: Opposite Mantri Commercio, Outer Ring Road, Devarabisanahalli, Near Sakra World Hospital
  rate  votes listed_in(city) listed_in(type)
 3.4/5      7       Bellandur        Delivery
3.4 /5      7   Sarjapur Road        Delivery

RESTAURANT: #L-81 Cafe
ADDRESS: Sector 6, HSR Layout, HSR
  rate  votes       listed_in(city) listed_in(type)
 3.9/5     48                   BTM        Delivery
 3.9/5     48                   BTM        Dine-out
 3.9/5     48                   HSR        Delivery
 3.9/5     48                   HSR        Dine-out
 3.9/5     48 Koramangala 4th Block        Delivery
3.9 /5     48 Koramangala 4th Block        Dine-out
3.9 /5     48 Koramangala 5th Block        Delivery
3.9 /5     48 Koramangala 6th Block        Delivery
3.9 /5     48 Koramangala 7th Block        Delivery

RESTAURANT: 1000 B.C
ADDRESS: 16, 17th A Main, Koramangala 5th Block, Bangalore
  rate  votes       listed_in(city) listed_in(type)
 3.2/5     49             

In [11]:
# Clean rating temporarily for investigation only
rate_clean_temp = (
    df["rate"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
)

rate_clean_temp = pd.to_numeric(rate_clean_temp, errors="coerce")

df["_rate_temp"] = rate_clean_temp

clean_rate_variation = (
    df.groupby(["name", "address"])["_rate_temp"]
      .nunique(dropna=False)
)

print(
    "Entities with varying ratings AFTER numeric extraction:",
    (clean_rate_variation > 1).sum()
)

print(
    "Percentage:",
    round(
        (clean_rate_variation > 1).sum()
        / len(clean_rate_variation) * 100,
        2
    )
)

Entities with varying ratings AFTER numeric extraction: 1624
Percentage: 12.99
